# 风速分布与功率曲线

这个示例比较两组平均风速接近、分布形状不同的风速序列。两组数据经过同一条非线性功率曲线后，平均出力并不相同。

示例中的分段曲线沿用本仓库案例使用的CF代理形式，只用于说明分布和非线性映射的关系，不代表具体风机的制造商功率曲线，也不用于真实AEP计算。

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
target_mean = 7.0
n = 200_000

def weibull_scale_for_mean(mean, k):
    return mean / math.gamma(1.0 + 1.0 / k)

k_a, k_b = 2.0, 5.0
c_a = weibull_scale_for_mean(target_mean, k_a)
c_b = weibull_scale_for_mean(target_mean, k_b)

ws_a = c_a * rng.weibull(k_a, n)
ws_b = c_b * rng.weibull(k_b, n)

print(f"A: mean={ws_a.mean():.3f} m/s, std={ws_a.std():.3f} m/s")
print(f"B: mean={ws_b.mean():.3f} m/s, std={ws_b.std():.3f} m/s")

两组样本的理论均值都设为7 m/s。k较小时分布更分散，k较大时风速更集中。后续比较分布形状对功率映射后统计量的影响。

In [ ]:
def cf_proxy(ws):
    ws = np.asarray(ws, dtype=float)
    out = np.zeros_like(ws)
    partial = (ws >= 3.0) & (ws < 12.0)
    rated = (ws >= 12.0) & (ws <= 25.0)
    out[partial] = ((ws[partial] - 3.0) / 9.0) ** 3
    out[rated] = 1.0
    return out

cf_a = cf_proxy(ws_a)
cf_b = cf_proxy(ws_b)

print(f"A: mean CF proxy={cf_a.mean():.4f}")
print(f"B: mean CF proxy={cf_b.mean():.4f}")

for label, ws, cf in [("A", ws_a, cf_a), ("B", ws_b, cf_b)]:
    cf_at_mean = float(cf_proxy(np.array([ws.mean()]))[0])
    print(
        f"{label}: E[CF(V)]={cf.mean():.4f}, "
        f"CF(E[V])={cf_at_mean:.4f}, "
        f"difference={cf.mean() - cf_at_mean:+.4f}"
    )

对应的关系为

$$
\mathbb{E}[P(V)]\neq P(\mathbb{E}[V]).
$$

平均风速接近，并不足以保证平均出力接近。风速分布在切入区、部分负荷区和额定区中的占比会影响最终结果。

In [ ]:
bins = np.linspace(0, 20, 60)

plt.figure(figsize=(8, 4.5))
plt.hist(ws_a, bins=bins, density=True, alpha=0.5, label=f"A: k={k_a}")
plt.hist(ws_b, bins=bins, density=True, alpha=0.5, label=f"B: k={k_b}")
plt.xlabel("Wind speed (m/s)")
plt.ylabel("Probability density")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
x = np.linspace(0, 30, 500)
y = cf_proxy(x)

plt.figure(figsize=(8, 4.5))
plt.plot(x, y)
plt.xlabel("Wind speed (m/s)")
plt.ylabel("CF proxy")
plt.tight_layout()
plt.show()

## 工程边界

真实风电项目还需要轮毂高度风况、空气密度、机组实际功率曲线、尾流、可利用率、电气损失、限电和长期不确定性。这个示例只保留“风速分布经过非线性功率映射”这一环。

参考：

- Lee & Fields (2021), https://doi.org/10.5194/wes-6-311-2021
- Drobinski (2026), https://doi.org/10.1038/s44168-025-00332-4